<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Large_Moves_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Volatility Breakout Strategy & Performance Scanner

This notebook implements a technical analysis scanner designed to identify **Volatility Breakouts**. The strategy focuses on identifying stocks that experience significant price movements (up or down) relative to their historical volatility.

**Key Features:**
*   **Dynamic Signal Detection:** Identifies trigger days where the absolute daily return exceeds a multiple of the historical rolling volatility (configurable via `VOL_MULTIPLIERS`).
*   **Cooldown Mechanism:** Prevents redundant signals by enforcing a 'cool-off' period (`COOLDOWN_DAYS`) after a trigger is detected for a specific ticker.
*   **Ticker Loading:** Loads ticker symbols from a Google Drive file or falls back to a predefined list.
*   **Historical Data Fetching:** Downloads historical stock data using `yfinance`.
*   **Forward Performance Tracking:** Automatically calculates forward returns (1-day through 5-day) following each breakout.
*   **Parameterized Retracement Analysis:** Analyzes the probability of price retracement back towards the previous day's close over the subsequent 5 days, customizable by a `TARGET_MULTIPLIER`.

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [9]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: df_symbols
Deleted DataFrame: raw_data
Deleted DataFrame: df_ticker
Deleted DataFrame: df_triggers
Deleted DataFrame: master_df
Deleted DataFrame: summary
Deleted DataFrame: df_filtered
Deleted DataFrame: df_daily_ret
Deleted DataFrame: summary_daily
All DataFrames cleared from memory.


In [10]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### Configure Scanner Parameters

In [11]:
# ---- SCANNER PARAMETERS ----
LOOKBACK_CALENDAR_DAYS = 100  # Pull enough data for the 60-day volatility window
SCAN_SIGNAL_DAYS = 20         # Filter final report to signals from the last 20 trading days
VOL_WINDOW = 60               # Rolling window for volatility
COOLDOWN_DAYS = 5             # Days to ignore new signals after a trigger
TARGET_MULTIPLIER = 3.0       # The breakout extreme we want to track profitability for
MAX_ALLOCATION = 5000         # Maximum dollar position size per trade

# Dynamically calculate window dates based on today
end_dt = datetime.today()
start_dt = end_dt - timedelta(days=LOOKBACK_CALENDAR_DAYS)

START_DATE = start_dt.strftime("%Y-%m-%d")
END_DATE = end_dt.strftime("%Y-%m-%d")

print(f"Scanner configured to pull historical data from {START_DATE} through today ({END_DATE}).")

Scanner configured to pull historical data from 2026-03-18 through today (2026-06-26).


### Load Tickers & Fetch Live Data

In [12]:
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'
fallback_tickers = ["AAPL", "MSFT", "NVDA", "AMD", "TSLA"]

try:
    print("Attempting to pull tickers from the OptionVolume file...")
    df_symbols = pd.read_csv(OptionVolume)
    if 'Symbol' in df_symbols.columns:
        tickers = df_symbols['Symbol'].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        print(f"Successfully loaded {len(tickers)} tickers from OptionVolume.")
    else:
        raise KeyError("'Symbol' column was not found in the downloaded file.")
except Exception as e:
    print(f"Failed to load tickers from Google Drive ({e}). Falling back to default list.")
    tickers = fallback_tickers

print(f"Fetching data block for {len(tickers)} tickers...")
# yf.download handles parsing the live intraday bar as today's 'Close' automatically
raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, group_by='ticker')

Attempting to pull tickers from the OptionVolume file...


/tmp/ipykernel_8510/693906147.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, group_by='ticker')
[                       0%                       ]

Successfully loaded 100 tickers from OptionVolume.
Fetching data block for 100 tickers...


[*********************100%***********************]  100 of 100 completed


### Live Signal Scan & Dollar Profitability Processing

In [21]:
# Master lists to hold results
scanner_results = []

for ticker in tickers:
    if ticker not in raw_data.columns.levels[0]:
        continue

    df_ticker = raw_data[ticker].dropna(subset=['Close']).copy()
    if len(df_ticker) < VOL_WINDOW:
        continue

    # Standard metrics
    df_ticker['Daily_Return'] = df_ticker['Close'].pct_change()
    df_ticker['Return_Mag'] = df_ticker['Daily_Return'].abs()
    df_ticker['Prev_Close'] = df_ticker['Close'].shift(1)
    df_ticker['Hist_Vol'] = df_ticker['Daily_Return'].rolling(window=VOL_WINDOW).std()
    df_ticker['Vol_Threshold'] = df_ticker['Hist_Vol'] * TARGET_MULTIPLIER

    # Process signals tracking cooldowns
    cooldown_counter = 0
    trigger_rows = []

    for idx, row in df_ticker.iterrows():
        if cooldown_counter > 0:
            cooldown_counter -= 1
            continue

        is_up = row['Daily_Return'] > 0 and row['Return_Mag'] > row['Vol_Threshold']
        is_down = row['Daily_Return'] < 0 and row['Return_Mag'] > row['Vol_Threshold']

        if is_up or is_down:
            direction = 'up' if is_up else 'down'
            trigger_rows.append((idx, direction))
            cooldown_counter = COOLDOWN_DAYS

    # Calculate forward dollar values for verified signals
    for idx, direction in trigger_rows:
        try:
            loc = df_ticker.index.get_loc(idx)
            day_0_close = df_ticker.iloc[loc]['Close']
            day_0_ret = df_ticker.iloc[loc]['Daily_Return']

            shares = MAX_ALLOCATION / day_0_close

            row_data = {
                'Date': idx.strftime("%Y-%m-%d"),
                'Symbol': ticker,
                'Price': round(day_0_close, 2),
                'Direction': direction,
                'Daily_Return': day_0_ret
            }

            for d in range(1, 6):
                if loc + d < len(df_ticker):
                    fwd_close = df_ticker.iloc[loc + d]['Close']
                    price_diff = fwd_close - day_0_close
                    dollar_profit = (price_diff * shares) if direction == 'up' else (-price_diff * shares)
                    row_data[f'Day_{d}_$'] = round(dollar_profit, 2)
                else:
                    row_data[f'Day_{d}_$'] = np.nan

            scanner_results.append(row_data)
        except Exception:
            continue

# Compile Master Scanner Output
if scanner_results:
    master_scanner_df = pd.DataFrame(scanner_results)
    master_scanner_df['Datetime'] = pd.to_datetime(master_scanner_df['Date'])
    master_scanner_df = master_scanner_df.sort_values(by='Datetime', ascending=False)

    # NEW LOGIC: Filter based on an actual rolling window of the last 20 trading days
    # We find the unique trading dates in our downloaded dataset and look at the last 20
    available_dates = sorted(master_scanner_df['Datetime'].unique())
    cutoff_date = available_dates[-min(SCAN_SIGNAL_DAYS, len(available_dates))]

    recent_signals_df = master_scanner_df[master_scanner_df['Datetime'] >= cutoff_date].copy()
    recent_signals_df = recent_signals_df.drop(columns=['Datetime'])

    total_found = len(recent_signals_df)
    if total_found > 0:
        print(f"Scan Complete: Processed {total_found} signals found within the true last {SCAN_SIGNAL_DAYS} trading days ({cutoff_date.strftime('%Y-%m-%d')} to today).")
    else:
        print(f"Scan Complete: 0 signals matched parameters within the last {SCAN_SIGNAL_DAYS} trading days.")
else:
    recent_signals_df = pd.DataFrame()
    print("Scan Complete: No signals generated across the historical timeline.")

Scan Complete: Processed 4 signals found within the true last 20 trading days (2026-06-15 to today).


### Generate Scanned Profitability Report

In [22]:
# ==========================================
# 5. GENERATE SCANNED RETRACEMENT REPORT
# ==========================================

print("📊" * 25)
print(f"CLOSING RETRACEMENT REPORT ({TARGET_MULTIPLIER}x Multiplier) — LAST {SCAN_SIGNAL_DAYS} SIGNALS")
print("📊" * 25)

if not recent_signals_df.empty:
    # Create a clean reporting copy to calculate percentages without mutating master data
    report_df = recent_signals_df.copy()

    for d in range(1, 6):
        dollar_col = f'Day_{d}_$'
        pct_col = f'Retrace_Day_{d}'

        # Absolute magnitude of the Day 0 baseline move
        impulse_size_pct = report_df['Daily_Return'].abs()

        # Percentage change from Day 0 close
        fwd_return_from_close = (report_df[dollar_col] / MAX_ALLOCATION)

        # Retracement % = -(Forward Return from Day 0 Close) / (Day 0 Return)
        report_df[pct_col] = -fwd_return_from_close / impulse_size_pct

    display_cols = ['Date', 'Symbol', 'Price', 'Direction', 'Retrace_Day_1', 'Retrace_Day_2', 'Retrace_Day_3', 'Retrace_Day_4', 'Retrace_Day_5']

    # Format floating numbers as percentages, leaving NaNs completely intact
    print(report_df[display_cols].to_string(index=False, na_rep='NaN', formatters={
        'Price': '${:,.2f}'.format,
        'Retrace_Day_1': '{:,.1%}'.format, 'Retrace_Day_2': '{:,.1%}'.format,
        'Retrace_Day_3': '{:,.1%}'.format, 'Retrace_Day_4': '{:,.1%}'.format, 'Retrace_Day_5': '{:,.1%}'.format
    }))
else:
    print(f"No breakout events crossed the {TARGET_MULTIPLIER}x standard deviation threshold in the last 20 days.")

📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
CLOSING RETRACEMENT REPORT (3.0x Multiplier) — LAST 20 SIGNALS
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
      Date Symbol     Price Direction Retrace_Day_1 Retrace_Day_2 Retrace_Day_3 Retrace_Day_4 Retrace_Day_5
2026-06-25   AAPL   $275.15      down           NaN           NaN           NaN           NaN           NaN
2026-06-25   SNDK $2,335.00        up           NaN           NaN           NaN           NaN           NaN
2026-06-25   AMAT   $668.00        up           NaN           NaN           NaN           NaN           NaN
2026-06-15    WDC   $653.53        up        -26.2%        -55.7%        -88.1%        -75.2%        -16.4%
